In [7]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Rohini_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,369.0,204.0,256.0,108.0,247.0,NaN,103.0,41.0,84.0,139.0,373.0,300.0
1,2,NaN,257.0,138.0,145.0,213.0,189.0,131.0,100.0,89.0,171.0,329.0,313.0
2,3,350.0,153.0,143.0,175.0,315.0,173.0,119.0,78.0,121.0,189.0,435.0,292.0
3,4,405.0,221.0,131.0,157.0,318.0,243.0,81.0,93.0,70.0,217.0,411.0,182.0
4,5,335.0,139.0,108.0,169.0,374.0,302.0,97.0,65.0,70.0,157.0,376.0,179.0
5,6,332.0,109.0,132.0,159.0,258.0,189.0,62.0,80.0,141.0,148.0,394.0,217.0
6,7,376.0,164.0,180.0,230.0,346.0,219.0,56.0,54.0,143.0,110.0,416.0,260.0
7,8,373.0,133.0,175.0,260.0,266.0,247.0,68.0,67.0,84.0,147.0,425.0,369.0
8,9,326.0,145.0,137.0,214.0,184.0,209.0,87.0,73.0,105.0,159.0,393.0,216.0
9,10,289.0,342.0,192.0,225.0,253.0,202.0,160.0,90.0,116.0,115.0,368.0,264.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    34 non-null     float64
 2   February   33 non-null     float64
 3   March      34 non-null     float64
 4   April      33 non-null     float64
 5   May        33 non-null     float64
 6   June       35 non-null     float64
 7   July       34 non-null     float64
 8   August     35 non-null     float64
 9   September  35 non-null     float64
 10  October    35 non-null     float64
 11  November   33 non-null     float64
 12  December   36 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [8]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [9]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [10]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [11]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,369.000000,204.0,256.0,108.0,247.000000,168.6,103.0,41.0,84.0,139.0,373.0,300.0
1,2,322.852941,257.0,138.0,145.0,213.000000,189.0,131.0,100.0,89.0,171.0,329.0,313.0
2,3,350.000000,153.0,143.0,175.0,315.000000,173.0,119.0,78.0,121.0,189.0,435.0,292.0
3,4,405.000000,221.0,131.0,157.0,318.000000,243.0,81.0,93.0,70.0,217.0,411.0,182.0
4,5,335.000000,139.0,108.0,169.0,222.909091,168.6,97.0,65.0,70.0,157.0,376.0,179.0
